# LLM-JEPA on Gemma: does a JEPA loss rescue LoRA when data is scarce?

One model, no encoder or bridge: **Gemma 3 4B IT + LoRA**, prompted with `Rewrite in the Titan's voice: <x>`, writes y in the Titan register.

| Arm (`ARM`) | Loss |
|---|---|
| **lora** | cross-entropy on y (plain LoRA fine-tune) |
| **jepa** | cross-entropy on y **+ λ · (1 − cos(E(x ⊕ [PRED]×k), sg E(y)))** |
| **jepa6** | cross-entropy on y **+ λ · (1 − cos(P(H_x), sg E(y)))**, where P is a 6-layer transformer predictor (width 512, 8 heads, one learned query) over the x view's layer states H_x, dropped at inference |

**The JEPA term**, in the style of LLM-JEPA (Huang, LeCun and Balestriero, 2025):
- **Two views of one pair:** the prompt with x, followed by k predictor tokens; and y on its own.
- **E(·):** Gemma's hidden state at the last token, taken from a **middle layer** (`JEPA_LAYER`), not the final one. The final layer is specialised for next-token prediction.
- **The predictor is Gemma itself:** the LoRA has to make the x view's state predict the y view's state. The y view has no gradient (stop-gradient), so it is the target.
- **Standardisation:** both views are standardised per dimension with statistics from the training targets, so a few outlier dimensions can't dominate the cosine.
- **The JEPA gradient trains the same LoRA that writes the output.**

**Why low data:** in notebook 12, plain LoRA on 1,260 pairs peaked at epoch 2 and then overfitted, and about half its outputs were judged usable. JEPA's claimed strength is generalisation from few examples. So both arms train on **300 pairs (30 per rhetorical move)** and are scored on the same 100 validation and 100 test sentences as notebook 12.

**Same in both arms:** model and precision (bf16), LoRA (r16 on all projections), learning rate, data and order, seed, early stopping on validation cross-entropy.

**Scores:** chrF, facts (numbers, dates, names, negation), the register's style rules, and a blind judgement of the test outputs done outside this notebook.

**The bar, set before the run:** jepa's blind "usable" rate at least **+10 points** over lora on test, and no worse on "nothing invented".

**Checkpoints:** saved after every epoch and mirrored to the private Hub repo `skamalj/titan-stage1` under `runs/<RUN_NAME>/`. A rerun resumes from them, and a finished arm is reloaded rather than retrained.

In [ ]:
# Cell 0 — Hugging Face token: HF_TOKEN in the environment (Modal) or a Kaggle secret; never printed

import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    for label in ("HF_TOKEN", "hf_token"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(label)
            break
        except Exception:
            pass
assert HF_TOKEN, "set HF_TOKEN (Modal secret) or add it under Kaggle Add-ons → Secrets"

## Cell 1 — Setup and settings

In [ ]:
# Cell 1 — Packages, settings, Hub mirror

!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import gc, json, random, re, shutil, statistics, time, urllib.request
from collections import Counter

import peft
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, hf_hub_download, login

DECODER = os.environ.get("DECODER", "google/gemma-3-4b-it")
ZERO_SHOT = os.environ.get("ZERO_SHOT") == "1"   # no training: prompt the base model only
DATA = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/project4-hindi-decoder/data/titan_v1/"
ARM = os.environ.get("ARM", "lora")              # "lora", "jepa" (predictor tokens) or "jepa6" (6-layer predictor)
N_PER_MOVE = int(os.environ.get("N_PER_MOVE", 30))   # 30 per move = 300 training pairs; 5 = 50
LAMBDA, K_PRED = 1.0, 2                          # JEPA weight, predictor tokens
JEPA_LAYER = None                                # None: the middle layer (n_layers // 2)
LR, PATIENCE, MICRO, ACCUM = 1e-4, 2, 4, 2
MAX_EPOCHS = int(os.environ.get("MAX_EPOCHS", 10))
MAX_NEW_TOKENS, MAX_TARGET_TOKENS = 128, 128
INSTRUCTIONS = {
    "titan": "Rewrite in the Titan's voice: ",
    "describe": ("Voice: calm, immense, patient; a being who has already decided. Never shouts, never jokes, rarely asks. "
                 "Short declaratives, one to five sentences, at most 60 words, no contractions, no exclamation marks. "
                 "Keep every fact of the sentence: times, names, numbers, the request. End on a settled note. "
                 "Rewrite this sentence in that voice and reply with the rewrite only: "),
    "thanos": "Rewrite this sentence in the voice of Thanos from the Marvel films, keeping every fact, and reply with the rewrite only: ",
}
INSTRUCTION = INSTRUCTIONS["titan"]           # the training prompt
SEED = 42
HUB_REPO, RUN_NAME = "skamalj/titan-stage1", "llm-jepa-v1"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DEVICE = torch.device("cuda:0")
assert ARM in ("lora", "jepa", "jepa6"), ARM
PRED_LAYERS, PRED_WIDTH, PRED_HEADS, PRED_LR = 6, 512, 8, 3e-4   # jepa6: the standard JEPA predictor
MODEL_TAG = DECODER.split("/")[-1].replace("gemma-3-", "").replace("-it", "")   # "4b", "1b"
TAG = ARM if N_PER_MOVE == 30 else f"{ARM}_n{10 * N_PER_MOVE}"   # file names; the 300-pair run keeps its own
if MODEL_TAG != "4b":
    TAG = f"{TAG}_{MODEL_TAG}"

login(HF_TOKEN)
api = HfApi(token=HF_TOKEN)
REV = api.model_info(DECODER, token=HF_TOKEN).sha
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), (transformers.__version__, peft.__version__)


def hub_path(local):
    return f"runs/{RUN_NAME}/{os.path.relpath(local, WORK)}"


def hub_put(local):
    try:
        if os.path.isdir(local):
            api.upload_folder(repo_id=HUB_REPO, folder_path=local, path_in_repo=hub_path(local),
                              commit_message=f"{RUN_NAME}: {hub_path(local)}")
        else:
            api.upload_file(repo_id=HUB_REPO, path_or_fileobj=local, path_in_repo=hub_path(local),
                            commit_message=f"{RUN_NAME}: {hub_path(local)}")
    except Exception as e:
        print(f"  (Hub mirror failed for {hub_path(local)}: {type(e).__name__}; the local copy is kept)")


def hub_get(local):
    if os.path.exists(local):
        return True
    remote = hub_path(local)
    try:
        files = [f for f in api.list_repo_files(HUB_REPO) if f == remote or f.startswith(remote + "/")]
        for f in files:
            dst = local if f == remote else os.path.join(local, os.path.relpath(f, remote))
            os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
            shutil.copy(hf_hub_download(HUB_REPO, f, token=HF_TOKEN), dst)
        return bool(files)
    except Exception as e:
        print(f"  (Hub restore failed for {remote}: {type(e).__name__})")
        return False


def save(obj, path, json_file=False):
    if json_file:
        json.dump(obj, open(path, "w"), ensure_ascii=False)
    else:
        torch.save(obj, path)
    hub_put(path)


print({"arm": ARM, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0), "gemma": REV, "work": WORK,
       "lambda": LAMBDA, "k_pred": K_PRED, "lr": LR, "n_per_move": N_PER_MOVE, "max_epochs": MAX_EPOCHS, "tag": TAG, "decoder": DECODER, "zero_shot": ZERO_SHOT})

## Cell 2 — Data: 300 training pairs (30 per move), the full validation and test sets

The subset is drawn with a fixed seed, so both arms train on the same 300 pairs. Validation and test are the same 100 + 100 sentences as in notebook 12.

In [ ]:
# Cell 2 — Data

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


FULL_TRAIN, VAL, TEST = fetch_jsonl("train.jsonl"), fetch_jsonl("validation.jsonl"), fetch_jsonl("test.jsonl")
rng = random.Random(SEED)
by_move = {}
for r in FULL_TRAIN:
    by_move.setdefault(r["move"], []).append(r)
TRAIN = []
for m in sorted(by_move):
    TRAIN += rng.sample(by_move[m], N_PER_MOVE)
TRAIN.sort(key=lambda r: r["id"])
print({"train": len(TRAIN), "validation": len(VAL), "test": len(TEST)}, "| moves:", dict(Counter(r["move"] for r in TRAIN)))
print("example:", TRAIN[0]["source"], "->", TRAIN[0]["target"])

## Cell 3 — Gemma 3 4B IT (bf16) + LoRA

In [ ]:
# Cell 3 — Model, tokenizer, LoRA, the predictor token

from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Gemma3ForConditionalGeneration

tok = AutoTokenizer.from_pretrained(DECODER, revision=REV, token=HF_TOKEN)
MULTIMODAL = MODEL_TAG not in ("1b", "270m")       # 4B+ are image-text models; 1B and 270M are text-only
LOADER = Gemma3ForConditionalGeneration if MULTIMODAL else AutoModelForCausalLM
model = LOADER.from_pretrained(DECODER, revision=REV, token=HF_TOKEN, dtype=torch.bfloat16, device_map={"": 0},
                               attn_implementation="eager")
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
torch.manual_seed(SEED)
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                         target_modules=rf"(.*language_model\.layers|model\.layers)\..*\.({'|'.join(PROJ)})"))
LORA = [p for n, p in model.named_parameters() if "lora_" in n]
EOS = tok.convert_tokens_to_ids("<end_of_turn>")
eos = model.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
N_LAYERS = getattr(model.config, "text_config", model.config).num_hidden_layers
LAYER = JEPA_LAYER if JEPA_LAYER is not None else N_LAYERS // 2
PRED = next((t for t in ("<unused0>", "<unused1>") if tok.convert_tokens_to_ids(t) not in (None, tok.unk_token_id)), None)
PRED_ID = tok.convert_tokens_to_ids(PRED) if PRED else tok.encode(" predict", add_special_tokens=False)[-1]
print(f"layers {N_LAYERS}, JEPA layer {LAYER} | predictor token {PRED or ' predict'} ({PRED_ID}) | "
      f"LoRA {sum(p.numel() for p in LORA) / 1e6:.1f} M")

## Cell 4 — Views, losses, generation, scores

- **SFT row:** chat prompt with x, then y and `<end_of_turn>`. The loss is on y only.
- **x view:** the chat prompt with x (ending in the model turn), then k predictor tokens. E is the hidden state of the last predictor token at the JEPA layer.
- **y view:** `<bos>` + y. E is the hidden state of its last token at the same layer, with no gradient.

In [ ]:
# Cell 4 — Views, losses, generation, scores

def prompt_ids(x, instruction=None):
    text = tok.apply_chat_template([{"role": "user", "content": (instruction or INSTRUCTION) + x}], tokenize=False,
                                   add_generation_prompt=True)
    return tok(text, add_special_tokens=False)["input_ids"]


def target_ids(y):
    return tok(y, add_special_tokens=False)["input_ids"][:MAX_TARGET_TOKENS]


def pad_batch(seqs, labels=None):
    T = max(len(s) for s in seqs)
    ids = torch.tensor([s + [tok.pad_token_id] * (T - len(s)) for s in seqs], device=DEVICE)
    mask = torch.tensor([[1] * len(s) + [0] * (T - len(s)) for s in seqs], device=DEVICE)
    lab = None if labels is None else torch.tensor([l + [-100] * (T - len(l)) for l in labels], device=DEVICE)
    return ids, mask, lab


def forward(ids, mask, labels=None, hidden=False):
    extra = {"token_type_ids": torch.zeros_like(mask)} if MULTIMODAL else {}
    return model(input_ids=ids, attention_mask=mask, labels=labels, output_hidden_states=hidden, **extra)


def ce_loss(records):
    seqs, labels = [], []
    for r in records:
        p, t = prompt_ids(r["source"]), target_ids(r["target"]) + [EOS]
        seqs.append(p + t)
        labels.append([-100] * len(p) + t)
    return forward(*pad_batch(seqs, labels)).loss


def last_hidden(seqs):
    ids, mask, _ = pad_batch(seqs)
    hs = forward(ids, mask, hidden=True).hidden_states[LAYER]
    return hs[torch.arange(len(seqs), device=DEVICE), mask.sum(1) - 1].float()


def x_view(records):
    return last_hidden([prompt_ids(r["source"]) + [PRED_ID] * K_PRED for r in records])


@torch.no_grad()
def y_view(records):
    return last_hidden([[tok.bos_token_id] + target_ids(r["target"]) for r in records])


STATS = {}


class Predictor(torch.nn.Module):
    """jepa6: a 6-layer pre-norm transformer over the x view's layer states plus one learned query; the query's output
    is the prediction of the standardised y-view state."""

    def __init__(self, d, width=PRED_WIDTH, layers=PRED_LAYERS, heads=PRED_HEADS, max_len=512):
        super().__init__()
        self.inp = torch.nn.Linear(d, width)
        self.pos = torch.nn.Parameter(torch.randn(max_len + 1, width) * 0.02)
        self.query = torch.nn.Parameter(torch.randn(1, width) * 0.02)
        layer = torch.nn.TransformerEncoderLayer(width, heads, 4 * width, dropout=0.1, activation="gelu",
                                                 batch_first=True, norm_first=True)
        self.body = torch.nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = torch.nn.LayerNorm(width)
        self.out = torch.nn.Linear(width, d)

    def forward(self, h, mask):
        B, T, _ = h.shape
        x = torch.cat([self.inp(h) + self.pos[:T], (self.query + self.pos[T]).expand(B, 1, -1)], 1)
        pad = torch.cat([mask == 0, torch.zeros(B, 1, dtype=torch.bool, device=h.device)], 1)
        return self.out(self.norm(self.body(x, src_key_padding_mask=pad)[:, -1]))


D_MODEL = getattr(model.config, "text_config", model.config).hidden_size
torch.manual_seed(SEED)
predictor = Predictor(D_MODEL).to(DEVICE) if ARM == "jepa6" else None


def x_states(records):
    """jepa6: the x view (prompt with x, no predictor tokens) at the JEPA layer, all positions."""
    ids, mask, _ = pad_batch([prompt_ids(r["source"]) for r in records])
    return forward(ids, mask, hidden=True).hidden_states[LAYER].float(), mask


def jepa_cos(records):
    zy = (y_view(records) - STATS["mean"]) / STATS["std"]
    if ARM == "jepa6":                        # the predictor speaks the standardised space directly
        return F.cosine_similarity(predictor(*x_states(records)), zy, dim=-1)
    return F.cosine_similarity((x_view(records) - STATS["mean"]) / STATS["std"], zy, dim=-1)


@torch.no_grad()
def val_losses(records, batch=8):
    model.eval()
    if predictor is not None:
        predictor.eval()
    ce = cos = 0.0
    for i in range(0, len(records), batch):
        b = records[i:i + batch]
        ce += ce_loss(b).item() * len(b)
        cos += jepa_cos(b).mean().item() * len(b)
    return {"validation_ce": round(ce / len(records), 4), "validation_jepa_cos": round(cos / len(records), 4)}


@torch.no_grad()
def translate(records, batch=16, instruction=None):
    model.eval()
    outs = []
    for i in range(0, len(records), batch):
        seqs = [prompt_ids(r["source"], instruction) for r in records[i:i + batch]]
        T = max(len(s) for s in seqs)                      # left padding for batched generation
        ids = torch.tensor([[tok.pad_token_id] * (T - len(s)) + s for s in seqs], device=DEVICE)
        mask = torch.tensor([[0] * (T - len(s)) + [1] * len(s) for s in seqs], device=DEVICE)
        gen = model.generate(input_ids=ids, attention_mask=mask, do_sample=False, num_beams=1,
                             max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS, pad_token_id=tok.pad_token_id,
                             temperature=None, top_p=None, top_k=None)
        for row in gen[:, T:].tolist():
            outs.append(tok.decode(row, skip_special_tokens=True).strip())
    return outs


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


# --- faithfulness checks: the fact checks of the dataset's own validator (raw/validate_titan.py), per row ---
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
ORDINAL = {1: "first", 2: "second", 3: "third", 5: "fifth", 8: "eighth", 9: "ninth", 12: "twelfth"}
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def cardinal(n):
    if n < 20:
        return ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return TENS[t] + ("" if o == 0 else "-" + ONES[o])
    if n < 1000:
        h, r = divmod(n, 100)
        return ONES[h] + " hundred" + ("" if r == 0 else " " + cardinal(r))
    if n < 1000000:
        th, r = divmod(n, 1000)
        return cardinal(th) + " thousand" + ("" if r == 0 else " " + cardinal(r))
    return str(n)


def ordinal(n):
    if n in ORDINAL:
        return ORDINAL[n]
    if n < 20:
        return cardinal(n) + "th"
    if n >= 100:
        head, tail = n - n % 100, n % 100
        return cardinal(head) + ("th" if tail == 0 else " " + ordinal(tail))
    if n % 10 == 0:
        return cardinal(n)[:-1] + "ieth"
    t, o = divmod(n, 10)
    return TENS[t] + "-" + ordinal(o)


def num_words(n):
    c, o = cardinal(n), ordinal(n)
    forms = {c, c.replace("-", " "), c.replace(" hundred ", " hundred and "), o, o.replace("-", " ")}
    if 1000 <= n < 10000 and n % 100 == 0:
        forms.add(cardinal(n // 100) + " hundred")
    forms |= {15: {"quarter past", "a quarter"}, 30: {"half past", "half an hour", "half"},
              45: {"quarter to"}}.get(n, set())
    return forms


has_any = lambda low, forms: any(f in low for f in forms)


def time_ok(h, mm, low):
    H, M = int(h), int(mm)
    nxt = H % 12 + 1
    return (f"{H}:{mm}" in low or (M == 0 and has_any(low, num_words(H)))
            or (has_any(low, num_words(H)) and has_any(low, num_words(M)))
            or (M == 15 and "quarter past" in low and has_any(low, num_words(H)))
            or (M == 30 and "half past" in low and has_any(low, num_words(H)))
            or (M == 45 and "quarter to" in low and has_any(low, num_words(nxt)))
            or (M > 30 and f"{cardinal(60 - M)} to" in low and has_any(low, num_words(nxt))))


def fact_checks(src, out):
    """Numbers (incl. clock times), weekdays/months and mid-sentence names of the source, found in the output."""
    low = out.lower().replace("-", " ")
    numbers, dates, names = [], [], []
    rest = src
    for m in re.finditer(r"\b(\d{1,2}):(\d{2})\b", src):
        numbers.append(time_ok(m.group(1), m.group(2), low))
        rest = rest.replace(m.group(0), " ")
    for m in re.finditer(r"\$?\d[\d,]*(?:\.\d+)?", rest):
        tok = m.group(0).replace("$", "").replace(",", "")
        if "." in tok:
            whole, frac = tok.split(".")
            numbers.append(tok in low or (tok == "0.25" and "quarter" in low) or (tok == "0.5" and "half" in low)
                           or (has_any(low, num_words(int(whole))) and has_any(low, num_words(int(frac)))))
        else:
            numbers.append(tok in low or has_any(low, num_words(int(tok))))
    for d in WEEKDAYS + MONTHS:
        present = (re.search(r"(\d\w*\s+May\b|\bMay\s+\d|\b(in|by|until|of|before|after|this|next|last)\s+May\b)", src)
                   if d == "may" else re.search(rf"\b{d}\b", src, re.I))
        if present:
            dates.append(d in low)
    for m in re.finditer(r"(?<![.?!]\s)(?<!^)\b([A-Z][a-z]{2,})\b", src):
        if m.group(1).lower() not in WEEKDAYS + MONTHS:
            names.append(m.group(1).lower() in low)
    return {"numbers_ok": all(numbers) if numbers else None, "dates_ok": all(dates) if dates else None,
            "names_ok": all(names) if names else None}


NEG = re.compile(r"\b(not|no|never|nobody|nothing|none|neither|nor|without|cannot|nowhere)\b|n't", re.I)
CONTRACTION = re.compile(r"\b(it|he|she|that|what|there|here|who|let|where|how|this)'s\b|\b\w+'(re|ve|ll|m)\b|n't\b"
                         r"|\b(i|he|she|we|they|you|it|that|who)'d\b", re.I)


def faithfulness(src, out):
    """Facts kept (the dataset's checks), negation kept, and the register's hard style rules."""
    n_sent = len([x for x in re.split(r"(?<=[.?!])\s+", out.strip()) if x])
    return {**fact_checks(src, out),
            "negation_ok": bool(NEG.search(out)) if NEG.search(src) else None,
            "style_ok": not CONTRACTION.search(out) and "!" not in out and len(out.split()) <= 60 and n_sent <= 5}


FAITH = ["numbers_ok", "dates_ok", "names_ok", "negation_ok"]          # facts: the verdict's faithfulness
STYLE = ["style_ok"]                                                   # the register's hard rules: reported only
norm = lambda t: re.sub(r"\W+", " ", t.lower()).strip()

train_targets = {r["target"].strip() for r in TRAIN}


def score(records, outputs):
    rows = [{"id": r["id"], "move": r["move"], "source": r["source"], "reference": r["target"], "output": o,
             "chrf": round(chrf(o, r["target"]), 2), **faithfulness(r["source"], o)} for r, o in zip(records, outputs)]
    s = {"n": len(rows), "chrf": round(statistics.mean(x["chrf"] for x in rows), 1),
         "distinct_outputs": len({x["output"] for x in rows}),
         "copies_of_source": sum(norm(x["output"]) == norm(x["source"]) for x in rows),
         "copies_of_training_targets": sum(x["output"].strip() in train_targets for x in rows)}
    for k in FAITH + STYLE:
        v = [x[k] for x in rows if x[k] is not None]
        s[k] = round(sum(v) / len(v), 3) if v else None
    return rows, s


# per-dimension statistics of the y view, from the base model (LoRA at zero), on the training targets
with torch.no_grad():
    model.eval()
    _z = torch.cat([y_view(TRAIN[i:i + 16]) for i in range(0, len(TRAIN), 16)])
    STATS.update(mean=_z.mean(0), std=_z.std(0).clamp_min(1e-3 * _z.std(0).mean().item()))
    del _z
print("y-view statistics ready | before training:", val_losses(VAL))

## Cell 4b — Zero-shot (only when `ZERO_SHOT=1`)

The base model, with no LoRA training, translates the 100 test sentences once per prompt:
- **titan:** the training prompt. The model cannot know the name; this is the true zero-shot baseline.
- **describe:** the style guide in two lines.
- **thanos:** the film character, named. This is a diagnostic of what the model already knows; nothing from it is used for training.

Scores are computed as for the trained arms, and the outputs go to `zeroshot_<model>.json`, mirrored to the Hub.

In [ ]:
# Cell 4b — Zero-shot

if ZERO_SHOT:
    ZS = {"decoder": DECODER, "prompts": INSTRUCTIONS, "results": {}, "scores": {}}
    for mode, instruction in INSTRUCTIONS.items():
        ZS["results"][mode], ZS["scores"][mode] = score(TEST, translate(TEST, instruction=instruction))
        print(f"[zero-shot {MODEL_TAG}] {mode}", ZS["scores"][mode])
    save(ZS, f"{WORK}/zeroshot_{MODEL_TAG}.json", json_file=True)
    for i in (0, 3, 9, 26):
        print(f"\nsrc: {TEST[i]['source']}")
        for mode in INSTRUCTIONS:
            print(f"  {mode:8s}: {ZS['results'][mode][i]['output'][:300]}")

## Cell 5 — Train one arm, score it, save it (checkpointed, mirrored)

- **Training:** micro-batches of 4, 2-step accumulation, up to 10 epochs. Each epoch has its own fixed data order. The learning rate decays linearly.
- **Early stopping:** after 2 epochs without a lower validation cross-entropy; the best epoch is kept.
- **Logged each epoch:** train CE and JEPA, validation CE and JEPA cosine (for both arms).
- **Files:**
  - `<arm>_ckpt.pt`: the LoRA, optimizer and log, after every epoch;
  - `<arm>_best.pt`: the best epoch's LoRA;
  - `<arm>_results.json`: the scores and every validation and test output.

In [ ]:
# Cell 5 — One arm

if not ZERO_SHOT:

    ck_file, best_file, out_file = f"{WORK}/{TAG}_ckpt.pt", f"{WORK}/{TAG}_best.pt", f"{WORK}/{TAG}_results.json"
    lora_state = lambda: [p.detach().to("cpu", copy=True) for p in LORA]


    def load_lora(state):
        for p, s in zip(LORA, state):
            p.data.copy_(s.to(p.device))


    if hub_get(out_file):
        RESULT = json.load(open(out_file))
        print(f"[{ARM}] already done; loaded {out_file}")
    else:
        for f in (ck_file, best_file):
            hub_get(f)
        steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
        groups = [{"params": LORA, "lr": LR}] + ([{"params": list(predictor.parameters()), "lr": PRED_LR}] if predictor else [])
        opt = torch.optim.AdamW(groups, weight_decay=0.0)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps))
        if os.path.exists(ck_file):
            ck = torch.load(ck_file, weights_only=False)
            load_lora(ck["lora"])
            if predictor is not None:
                predictor.load_state_dict(ck["predictor"])
            opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"])
            log, best, since = ck["log"], ck["best"], ck["since"]
            print(f"[{ARM}] resuming after epoch {log[-1]['epoch']}")
        else:
            log = [{"epoch": 0, **val_losses(VAL)}]
            best, since = (log[0]["validation_ce"], 0), 0
            save(lora_state(), best_file)
            print(f"[{ARM}]", log[-1])
        t0 = time.perf_counter()
        for epoch in range(log[-1]["epoch"] + 1, MAX_EPOCHS + 1):
            if since >= PATIENCE:
                break
            torch.manual_seed(SEED + epoch)
            order = random.Random(SEED + epoch).sample(TRAIN, len(TRAIN))
            batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
            run_ce, run_j = [], []
            model.train()
            if predictor is not None:
                predictor.train()
            for n, b in enumerate(batches, 1):
                ce = ce_loss(b)
                if ARM in ("jepa", "jepa6"):
                    j = (1 - jepa_cos(b)).mean()
                    loss = ce + LAMBDA * j
                    run_j.append(j.item())
                else:
                    loss = ce
                (loss / ACCUM).backward()
                run_ce.append(ce.item())
                if n % ACCUM == 0 or n == len(batches):
                    opt.step()
                    sched.step()
                    opt.zero_grad()
            log.append({"epoch": epoch, "train_ce": round(statistics.mean(run_ce), 4),
                        **({"train_jepa": round(statistics.mean(run_j), 4)} if run_j else {}), **val_losses(VAL)})
            print(f"[{ARM}]", log[-1], f"| {time.perf_counter() - t0:.0f}s")
            if log[-1]["validation_ce"] < best[0]:
                best, since = (log[-1]["validation_ce"], epoch), 0
                save(lora_state(), best_file)
            else:
                since += 1
            save({"lora": lora_state(), "opt": opt.state_dict(), "sched": sched.state_dict(), "log": log, "best": best,
                  "since": since, "predictor": predictor.state_dict() if predictor is not None else None}, ck_file)
            if since >= PATIENCE:
                print(f"[{ARM}] early stop after epoch {epoch}")
        load_lora(torch.load(best_file, weights_only=False))
        print(f"[{ARM}] kept epoch {best[1]} (validation cross-entropy {best[0]})")
        del opt, sched
        gc.collect()
        torch.cuda.empty_cache()
        RESULT = {"arm": ARM, "settings": {"n_train": len(TRAIN), "lambda": LAMBDA, "k_pred": K_PRED, "layer": LAYER,
                                               "predictor": [PRED_LAYERS, PRED_WIDTH, PRED_HEADS, PRED_LR] if predictor else None,
                                           "lr": LR, "seed": SEED, "max_epochs": MAX_EPOCHS}, "log": log, "kept_epoch": best[1],
                  "results": {}, "scores": {}}
        for name, recs in (("validation", VAL), ("test", TEST)):
            RESULT["results"][name], RESULT["scores"][name] = score(recs, translate(recs))
            print(f"[{ARM}] {name}", RESULT["scores"][name])
        model.save_pretrained(f"{WORK}/lora-{TAG}")
        hub_put(f"{WORK}/lora-{TAG}")
        save(RESULT, out_file, json_file=True)
    for x in RESULT["results"]["test"][:8]:
        print(f"\nsrc: {x['source']}\nout: {x['output']}\nref: {x['reference']}")